# Task 038: mrf-reconstruction-mlmir2020-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: MR fingerprinting parameter mapping using invertible neural network

**Paper**: Learning Bloch Simulations for MR Fingerprinting by Invertible Neural Networks
**Repository**: https://github.com/fabianbalsiger/mrf-reconstruction-mlmir2020

### Physical Background

This inverse problem recovers unknowns from indirect, noisy measurements using a forward model with regularization.

### Forward Model

```
y = A(x) + n
```

### Inverse Problem

```
x_hat = argmin ||A(x) - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: N/A (MRF — tissue property matching accuracy)
- **SSIM**: N/A
- **Evaluation**: MR fingerprinting matching accuracy

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import matplotlib.pyplot as plt

# =============================================================================
# DEFINITIONS & CONSTANTS
# =============================================================================

KEY_FINGERPRINTS = 'fingerprints'
KEY_MR_PARAMS = 'mr_params'

ID_MAP_FF = 'FFmap'
ID_MAP_T1H2O = 'T1H2Omap'
ID_MAP_T1FAT = 'T1FATmap'
ID_MAP_B0 = 'B0map'
ID_MAP_B1 = 'B1map'

MR_PARAMS = (ID_MAP_FF, ID_MAP_T1H2O, ID_MAP_T1FAT, ID_MAP_B0, ID_MAP_B1)

FILE_NAME_FINGERPRINTS = 'fingerprints.npy'
FILE_NAME_PARAMETERS = 'parameters.npy'
FILE_NAME_PARAMETERS_MIN = 'parameters_mins.pkl'
FILE_NAME_PARAMETERS_MAX = 'parameters_maxs.pkl'

# =============================================================================
# HELPER CLASSES & FUNCTIONS (MODEL & UTILS)
# =============================================================================

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`de_normalize`, `de_normalize_mr_parameters`, `get_invnet`, `load_and_preprocess_data`

In [ ]:
def de_normalize(data: np.ndarray, minmax_tuple: tuple):
    return data * (minmax_tuple[1] - minmax_tuple[0]) + minmax_tuple[0]

def de_normalize_mr_parameters(data: np.ndarray, mr_param_ranges, mr_params=MR_PARAMS):
    data_de_normalized = data.copy()
    for idx, mr_param in enumerate(mr_params):
        if mr_param in mr_param_ranges:
             data_de_normalized[:, idx] = de_normalize(data[:, idx], mr_param_ranges[mr_param])
    return data_de_normalized

def get_invnet(ndim, nb_blocks=4, hidden_layer=128, permute=True):
    modules = []
    for i in range(nb_blocks):
        def subnet_constructor(ch_in, ch_out):
            return F_fully_connected_small(ch_in, ch_out, internal_size=hidden_layer)
        modules.append(RNVPCouplingBlock([ndim], subnet_constructor))
        if permute:
            modules.append(PermuteRandom([ndim], seed=i))
    return SequenceINN(*modules)

def load_and_preprocess_data(data_dir: str, batch_size: int, device_str: str):
    """
    Loads the MRF dataset, creates a dataloader, determines dimensions,
    and initializes the Invertible Neural Network.
    
    Returns:
        tuple: (dataloader, model, optimizer, dims_dict, dataset_obj, device)
    """
    if not os.path.exists(data_dir):
        # Create dummy data for demonstration if folder doesn't exist
        # This ensures the code runs without requiring external large files
        os.makedirs(data_dir, exist_ok=True)
        N_SAMPLES = 100
        DIM_PARAMS = 5
        DIM_FINGER = 100
        
        dummy_params = np.random.rand(N_SAMPLES, DIM_PARAMS).astype(np.float32)
        dummy_fingers = np.random.randn(N_SAMPLES, DIM_FINGER).astype(np.float32)
        # Normalize fingerprints L2
        dummy_fingers = dummy_fingers / np.linalg.norm(dummy_fingers, axis=1, keepdims=True)
        
        np.save(os.path.join(data_dir, FILE_NAME_PARAMETERS), dummy_params)
        np.save(os.path.join(data_dir, FILE_NAME_FINGERPRINTS), dummy_fingers)
        
        mins = {k: 0.0 for k in MR_PARAMS}
        maxs = {k: 1.0 for k in MR_PARAMS}
        with open(os.path.join(data_dir, FILE_NAME_PARAMETERS_MIN), 'wb') as f:
            pickle.dump(mins, f)
        with open(os.path.join(data_dir, FILE_NAME_PARAMETERS_MAX), 'wb') as f:
            pickle.dump(maxs, f)
            
        print(f"Created dummy dataset in {data_dir}")

    device = torch.device(device_str if torch.cuda.is_available() else 'cpu')
    
    dataset = NumpyMRFDataset(data_dir)
    dataloader = data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Get dimensions
    sample = dataset[0]
    ndim_x = sample[KEY_MR_PARAMS].shape[0]
    ndim_y = sample[KEY_FINGERPRINTS].shape[0]
    
    dims = {'ndim_x': ndim_x, 'ndim_y': ndim_y}
    
    model = get_invnet(ndim=ndim_y, nb_blocks=4, hidden_layer=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    return dataloader, model, optimizer, dims, dataset, device

## 4. Class: `NumpyMRFDataset`

Core algorithm class.

Methods: `__init__`, `__len__`, `__getitem__`

In [ ]:
class NumpyMRFDataset(data.Dataset):
    def __init__(self, dataset_dir: str, index_selection: list = None, transform=None) -> None:
        super().__init__()
        # Ensure files exist to avoid silent failures later
        if not os.path.exists(os.path.join(dataset_dir, FILE_NAME_PARAMETERS)):
             raise FileNotFoundError(f"Parameters file not found in {dataset_dir}")

        self.mr_params = np.load(os.path.join(dataset_dir, FILE_NAME_PARAMETERS), mmap_mode='r')
        self.fingerprints = np.load(os.path.join(dataset_dir, FILE_NAME_FINGERPRINTS), mmap_mode='r')
        
        with open(os.path.join(dataset_dir, FILE_NAME_PARAMETERS_MIN), 'rb') as f:
            mins = pickle.load(f)
        with open(os.path.join(dataset_dir, FILE_NAME_PARAMETERS_MAX), 'rb') as f:
            maxs = pickle.load(f)
            
        self.mr_param_ranges = {k: (mins[k], maxs[k]) for k in mins}

        if index_selection is not None:
            indexes = np.asarray([int(k) for k in index_selection])
            indexes.sort()
        else:
            indexes = np.arange(self.mr_params.shape[0])
        self.indexes = indexes
        self.transform = transform

    def __len__(self) -> int:
        return self.indexes.shape[0]

    def __getitem__(self, index: int):
        mr_p = np.asarray(self.mr_params[self.indexes[index]]).copy()
        fp = np.asarray(self.fingerprints[self.indexes[index]]).copy()
        
        sample = {KEY_MR_PARAMS: mr_p.astype(np.float32),
                  KEY_FINGERPRINTS: fp.astype(np.float32)}
        if self.transform:
            sample = self.transform(sample)
        return sample

## 5. Class: `InvertibleModule`

Core algorithm class.

Methods: `forward`

In [ ]:
class InvertibleModule(nn.Module):
    def forward(self, x, rev=False):
        pass

## 6. Class: `SequenceINN`

Core algorithm class.

Methods: `__init__`, `forward`

In [ ]:
class SequenceINN(InvertibleModule):
    def __init__(self, *modules):
        super().__init__()
        self.module_list = nn.ModuleList(modules)

    def forward(self, x, rev=False):
        if not rev:
            for mod in self.module_list:
                x = mod(x, rev=False)
        else:
            for mod in reversed(self.module_list):
                x = mod(x, rev=True)
        return x

## 7. Class: `F_fully_connected_small`

Core algorithm class.

Methods: `__init__`, `forward`

In [ ]:
class F_fully_connected_small(nn.Module):
    def __init__(self, size_in, size, internal_size=None, dropout=0.0):
        super(F_fully_connected_small, self).__init__()
        if not internal_size:
            internal_size = 2*size

        self.d1 = nn.Dropout(p=dropout)
        self.fc1 = nn.Linear(size_in, internal_size)
        self.fc3 = nn.Linear(internal_size, size)
        self.nl1 = nn.ReLU()

    def forward(self, x):
        out = self.nl1(self.d1(self.fc1(x)))
        out = self.fc3(out)
        return out

## 8. Class: `RNVPCouplingBlock`

Core algorithm class.

Methods: `__init__`, `forward`

In [ ]:
class RNVPCouplingBlock(InvertibleModule):
    def __init__(self, dims_in, subnet_constructor):
        super().__init__()
        self.ndims = dims_in[0]
        self.split_len1 = self.ndims // 2
        self.split_len2 = self.ndims - self.split_len1
        
        self.subnet1 = subnet_constructor(self.split_len1, self.split_len2 * 2)

    def forward(self, x, rev=False):
        if not rev:
            x1, x2 = x[:, :self.split_len1], x[:, self.split_len1:]
            out = self.subnet1(x2)
            s1, t1 = out[:, :self.split_len2], out[:, self.split_len2:]
            s1 = torch.clamp(s1, -15.0, 15.0) 
            y1 = x1 * torch.exp(s1) + t1
            y2 = x2
            return torch.cat([y1, y2], dim=1)
        else:
            y1, y2 = x[:, :self.split_len1], x[:, self.split_len1:]
            x2 = y2
            out = self.subnet1(x2)
            s1, t1 = out[:, :self.split_len2], out[:, self.split_len2:]
            s1 = torch.clamp(s1, -15.0, 15.0)
            x1 = (y1 - t1) * torch.exp(-s1)
            return torch.cat([x1, x2], dim=1)

## 9. Class: `PermuteRandom`

Core algorithm class.

Methods: `__init__`, `forward`

In [ ]:
class PermuteRandom(InvertibleModule):
    def __init__(self, dims_in, seed=None):
        super().__init__()
        self.in_channels = dims_in[0]
        if seed is not None:
            np.random.seed(seed)
        self.perm = np.random.permutation(self.in_channels)
        self.perm_inv = np.zeros_like(self.perm)
        for i, p in enumerate(self.perm):
            self.perm_inv[p] = i
        
        self.register_buffer('perm_tensor', torch.LongTensor(self.perm))
        self.register_buffer('perm_inv_tensor', torch.LongTensor(self.perm_inv))

    def forward(self, x, rev=False):
        if not rev:
            return x[:, self.perm_tensor]
        else:
            return x[:, self.perm_inv_tensor]

## 10. Forward Model

Define the forward operator mapping true model to observations.

```
y = A(x) + n
```

Functions: `forward_operator`

In [ ]:
def forward_operator(model, x, ndim_y, device):
    """
    Simulates the forward Bloch process (learning-based).
    Maps parameters x -> fingerprint y.
    
    Because the INN has fixed input/output dimension equal to ndim_y,
    and x usually has fewer dimensions (ndim_x), we pad x with zeros.
    """
    if not isinstance(x, torch.Tensor):
        x = torch.from_numpy(x).float()
    
    x = x.to(device)
    if x.ndim == 1:
        x = x.unsqueeze(0)
        
    current_bs = x.size(0)
    ndim_x = x.size(1)
    
    pad_len = ndim_y - ndim_x
    if pad_len < 0:
        raise ValueError("Parameter dimension cannot be larger than fingerprint dimension for this INN architecture.")
        
    pad_x = torch.zeros(current_bs, pad_len, device=device)
    x_padded = torch.cat((x, pad_x), dim=1)
    
    # Forward pass through INN (x -> y)
    y_pred = model(x_padded, rev=False)
    
    return y_pred

## 11. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
x_hat = argmin ||A(x) - y||^2 + lambda R(x)
```

Functions: `run_inversion`

In [ ]:
def run_inversion(model, y, ndim_x, device):
    """
    Performs the inversion (Reconstruction).
    Maps fingerprint y -> parameters x.
    """
    if not isinstance(y, torch.Tensor):
        y = torch.from_numpy(y).float()
        
    y = y.to(device)
    if y.ndim == 1:
        y = y.unsqueeze(0)
    
    # Inverse pass through INN (y -> x_padded)
    x_rec_padded = model(y, rev=True)
    
    # Crop the padding to get the actual parameters
    x_rec = x_rec_padded[:, :ndim_x]
    
    return x_rec.detach()

## 12. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(model, dataloader, dataset, dims, device, epochs):
    """
    Trains the INN model and then evaluates it on a sample.
    """
    ndim_x = dims['ndim_x']
    ndim_y = dims['ndim_y']
    
    # -----------------------
    # TRAINING LOOP
    # -----------------------
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    model.train()
    
    print(f"Starting training for {epochs} epochs...")
    for epoch in range(epochs):
        total_loss = 0
        for batch in dataloader:
            x = batch[KEY_MR_PARAMS].to(device)
            y = batch[KEY_FINGERPRINTS].to(device)
            current_bs = x.size(0)
            
            # Pad x to match y dimension for INN
            pad_x = torch.zeros(current_bs, ndim_y - ndim_x, device=device)
            x_padded = torch.cat((x, pad_x), dim=1)
            
            optimizer.zero_grad()
            
            # Forward loss: predict y from x
            y_hat = model(x_padded, rev=False)
            loss_fwd = F.mse_loss(y_hat, y)
            
            # Backward loss: predict x from y
            x_hat_padded = model(y, rev=True)
            loss_bwd = F.mse_loss(x_hat_padded, x_padded)
            
            loss = loss_fwd + loss_bwd
            loss.backward()
            
            # Gradient Clipping
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.data.clamp_(-15.00, 15.00)
            
            optimizer.step()
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.6f}")

    # -----------------------
    # EVALUATION
    # -----------------------
    model.eval()
    
    # Get a single sample for evaluation
    sample_idx = 0
    sample = dataset[sample_idx]
    x_gt_np = sample[KEY_MR_PARAMS]
    y_gt_np = sample[KEY_FINGERPRINTS]
    
    # 1. Inversion (y -> x)
    x_rec = run_inversion(model, y_gt_np, ndim_x, device)
    x_rec_np = x_rec.cpu().numpy()
    
    # 2. Forward (x -> y)
    y_pred = forward_operator(model, x_gt_np, ndim_y, device)
    y_pred_np = y_pred.detach().cpu().numpy()

    # 3. Denormalize parameters for display
    x_gt_denorm = de_normalize_mr_parameters(x_gt_np[np.newaxis, :], dataset.mr_param_ranges)
    x_rec_denorm = de_normalize_mr_parameters(x_rec_np, dataset.mr_param_ranges)
    
    print("\nReconstruction Results (Parameters):")
    param_names = MR_PARAMS
    for i in range(ndim_x):
        name = param_names[i] if i < len(param_names) else f"Param {i}"
        err = abs(x_gt_denorm[0,i] - x_rec_denorm[0,i])
        print(f"{name}: GT = {x_gt_denorm[0,i]:.4f}, Pred = {x_rec_denorm[0,i]:.4f}, Error = {err:.4f}")
        
    mse_fp = np.mean((y_pred_np - y_gt_np)**2)
    print(f"\nForward Model Fingerprint MSE: {mse_fp:.6f}")
    
    plt.figure(figsize=(10, 4))
    plt.plot(y_gt_np, label='Ground Truth')
    plt.plot(y_pred_np[0], label='Predicted (Learned Bloch)')
    plt.title('MR Fingerprint Comparison')
    plt.legend()
    plt.grid(True)
    plt.savefig('mrf_fingerprint_comparison.png')
    print("Saved fingerprint comparison plot to mrf_fingerprint_comparison.png")
    
    return x_rec_denorm

## 13. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Configuration
data_directory = 'in/train' 
batch_size = 50
device_name = 'cuda'
num_epochs = 2

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
dl, net, opt, dimensions, ds_obj, dev = load_and_preprocess_data(data_directory, batch_size, device_name)

# Note: Training is part of the evaluation logic in this specific refactor 
# because the original code mixed training and evaluation in main.
# To adhere to the constraints strictly, we treat "run_inversion" and "forward_operator"
# as the core functional calls used INSIDE "evaluate_results" after training.

### 4. Evaluate (includes Training Loop inside to setup the model state)

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate (includes Training Loop inside to setup the model state)
final_params = evaluate_results(net, dl, ds_obj, dimensions, dev, num_epochs)

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 14. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **mrf-reconstruction-mlmir2020-master**:

1. **Problem Setup**: This inverse problem recovers unknowns from indirect, noisy measurements using a forward model with regularization.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=N/A (MRF — tissue property matching accuracy), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Learning Bloch Simulations for MR Fingerprinting by Invertible Neural Networks
- Repository: https://github.com/fabianbalsiger/mrf-reconstruction-mlmir2020
